# 통합인사정보 데이터 전처리

### 라이브러리 설치

In [2]:
!pip install pandas python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 라이브러리 선언

In [3]:
import pandas as pd
import re
from datetime import date
from pathlib import Path
import os
from dotenv import load_dotenv

print(f'pandas 버전: {pd.__version__}')
print('라이브러리 로딩 완료!')

pandas 버전: 3.0.3
라이브러리 로딩 완료!


### 환경 설정 *** 경로 및 설정값 확인 필요

In [ ]:
load_dotenv()

INPUT_DIR  = Path(os.getenv('INPUT_DIR',  'dataset'))
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'output'))

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('경로 설정:')
print(f'  입력 디렉토리: {INPUT_DIR}')
print(f'  출력 디렉토리: {OUTPUT_DIR}')
print(f'\n출력 파일 예시:')
print(f"  {OUTPUT_DIR / '통합인사정보_정제.csv'}")
print(f"  {OUTPUT_DIR / 'error_log.csv'}")

In [ ]:
TODAY = date.today()

# 이상치 허용 범위
MIN_AGE,    MAX_AGE    = 18, 80
MIN_SALARY, MAX_SALARY = 20_000_000, 500_000_000
MAX_OVERTIME           = 52
MAX_UNUSED_LEAVE       = 30
MIN_GPA,    MAX_GPA    = 0.0, 4.5
MAX_SCORE              = 100
MAX_TOEIC              = 990

# 조직 고정값
DEPARTMENTS = ['개발부', '인사부', '영업부', '마케팅부', '기획부']

DEPT_TEAM_MAP = {
    '개발부':   ['백엔드팀', '프론트팀', 'AI팀', '인프라팀'],
    '인사부':   ['채용팀', '교육팀'],
    '영업부':   ['국내영업팀', '해외영업팀'],
    '마케팅부': ['디지털마케팅팀', '브랜드팀'],
    '기획부':   ['전략기획팀', '사업기획팀'],
}

DEPT_LEVEL_MAP = {
    '개발부':   1,
    '인사부':   3,
    '영업부':   1,
    '마케팅부': 1,
    '기획부':   1,
}

GRADE_LEVEL_MAP = {
    '사원': 1, '대리': 1, '과장': 1,
    '차장': 2, '부장': 2,
    '이사': 3, '사장': 3,
}

POSITIONS        = ['팀원', '팀장', '본부장', '대표이사']
PERF_GRADES      = ['S', 'A', 'B', 'C', 'D', 'F']
INSURANCE_VALUES = ['가입', '미가입']
SUBSIDY_VALUES   = ['해당', '비해당']
PERF_YEARS       = [2020, 2021, 2022, 2023, 2024]

print('설정된 허용 범위:')
print(f'  나이        : {MIN_AGE} ~ {MAX_AGE}세')
print(f'  연봉        : {MIN_SALARY:,} ~ {MAX_SALARY:,}원')
print(f'  잔업시간    : 0 ~ {MAX_OVERTIME}시간')
print(f'  미사용휴가  : 0 ~ {MAX_UNUSED_LEAVE}일')
print(f'  학점        : {MIN_GPA} ~ {MAX_GPA}')
print(f'  성과점수    : 0 ~ {MAX_SCORE}')
print(f'  TOEIC       : 0 ~ {MAX_TOEIC}')
print(f'\n조직 고정값:')
print(f'  부서        : {DEPARTMENTS}')
print(f'  직책        : {POSITIONS}')
print(f'  인사고과    : {PERF_GRADES}')

# 1. 데이터 로딩

In [ ]:
INPUT_PATH = INPUT_DIR / '통합인사정보.csv'
df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')

print('로딩 완료!')
print(f'  파일: {INPUT_PATH}')
print(f'  행 수: {len(df):,}')
print(f'  열 수: {len(df.columns)}')
print(f'\n컬럼 목록:')
print(list(df.columns))

In [ ]:
print('첫 5행 미리보기:')
print('-' * 50)
df.head()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'결측수': missing, '결측율(%)': missing_pct})
missing_summary = missing_summary[missing_summary['결측수'] > 0]

print(f'결측치 있는 컬럼 수: {len(missing_summary)}개')
print('-' * 40)
print(missing_summary)

### 에러 로그 초기화

In [ ]:
# 에러 내용을 저장하는 리스트
_errors = []
# 삭제할 행 번호를 저장하는 집합
drop_rows = set()

def log(row, emp_id, column, original_value, reason):
    _errors.append({
        '행': row,
        '사원번호': emp_id,
        '컬럼': column,
        '원본값': original_value,
        '사유': reason
    })

print('에러 로그 초기화 완료!')

### 헬퍼 함수 정의

In [ ]:
# 만 나이 계산
def calc_age(birth):
    age = TODAY.year - birth.year
    if (TODAY.month, TODAY.day) < (birth.month, birth.day):
        age -= 1
    return age

# 근속기간(년) 계산
def tenure_years(hire_date):
    days = (TODAY - hire_date).days
    return round(days / 365.25, 2)

# 문자열 → 날짜 변환 (실패 시 None)
def parse_date(text):
    try:
        text = str(text).strip()
        if not text:
            return None
        return pd.to_datetime(text).date()
    except Exception:
        return None

# 주민번호 앞6자리 → 생년월일
def birth_from_rrn(rrn):
    try:
        digits = rrn.replace('-', '')
        yy = int(digits[0:2])
        mm = int(digits[2:4])
        dd = int(digits[4:6])
        seventh = int(digits[6])
        if seventh in (1, 2):
            year = 1900 + yy
        else:
            year = 2000 + yy
        return date(year, mm, dd)
    except Exception:
        return None

# 주민번호 체크섬 검증
def rrn_checksum_ok(rrn):
    digits = rrn.replace('-', '')
    if len(digits) != 13 or not digits.isdigit():
        return False
    weights = [2, 3, 4, 5, 6, 7, 8, 9, 2, 3, 4, 5]
    total = 0
    for i in range(12):
        total += int(digits[i]) * weights[i]
    check = (11 - total % 11) % 10
    return check == int(digits[12])

# 숫자/특수문자만으로 구성됐는지 확인
def only_special(text):
    return bool(re.fullmatch(r'[^가-힣a-zA-Z]+', text.strip()))

# 쉼표 구분 배열 파싱
def parse_array(cell_value):
    if pd.isna(cell_value) or not str(cell_value).strip():
        return []
    items = []
    for item in str(cell_value).split(','):
        item = item.strip()
        if item:
            items.append(item)
    return items

# 숫자 변환 (실패 시 에러 로그 후 None)
def to_num(cell_value, column, row, emp_id):
    try:
        return float(cell_value)
    except Exception:
        log(row, emp_id, column, cell_value, '숫자 변환 불가')
        return None

# 현재 행의 사원번호 조회
def get_emp_id(row):
    cell_value = df.at[row, '사원번호']
    if pd.notna(cell_value):
        return str(cell_value).strip()
    return str(row)

print('헬퍼 함수 정의 완료!')

# 2. 데이터 정제

### 2-1. 식별자 (사원번호 · 이름)

In [ ]:
print('사원번호 정제 시작...')
pattern_emp = re.compile(r'^EMP\d{4}$')
seen_emp = {}

for row, raw in df['사원번호'].items():
    if pd.notna(raw):
        emp_id = str(raw).strip()
    else:
        emp_id = ''

    if not emp_id:
        log(row, emp_id, '사원번호', emp_id, '결측')
        drop_rows.add(row)
    elif not pattern_emp.match(emp_id):
        log(row, emp_id, '사원번호', emp_id, '형식 오류 (EMP+4자리)')
        drop_rows.add(row)
    elif emp_id in seen_emp:
        log(row, emp_id, '사원번호', emp_id, '중복 → 나중 행 제거')
        drop_rows.add(row)
    else:
        seen_emp[emp_id] = row

    df.at[row, '사원번호'] = emp_id

print('사원번호 정제 완료!')
print(f'  제거 대상 행: {len(drop_rows)}개')
print(f'  정상 사원번호: {len(seen_emp)}개')

In [ ]:
print('이름 정제 시작...')
name_err = 0

for row, raw in df['이름'].items():
    if pd.notna(raw):
        name = str(raw).strip()
    else:
        name = ''

    if not name:
        log(row, get_emp_id(row), '이름', name, '결측')
        drop_rows.add(row)
        name_err += 1
    elif not re.fullmatch(r'[가-힣a-zA-Z一-鿿\s]+', name):
        log(row, get_emp_id(row), '이름', name, '숫자/특수문자 포함')
        drop_rows.add(row)
        name_err += 1

print('이름 정제 완료!')
print(f'  오류 건수: {name_err}건')

### 2-2. 개인정보 (주민번호 · 성별 · 나이 · 생년월일 · 병역)

In [ ]:
print('주민등록번호 정제 시작...')
rrn_pat = re.compile(r'^\d{6}-\d{7}$')
valid_rrn = {}
rrn_err = 0

for row, raw in df['주민등록번호'].items():
    if pd.notna(raw):
        rrn = str(raw).strip()
    else:
        rrn = ''
    emp_id = get_emp_id(row)

    if not rrn:
        log(row, emp_id, '주민등록번호', rrn, '결측')
        df.at[row, '주민등록번호'] = '미입력'
        rrn_err += 1
        continue

    if not rrn_pat.match(rrn):
        log(row, emp_id, '주민등록번호', rrn, '형식 오류')
        df.at[row, '주민등록번호'] = '미입력'
        rrn_err += 1
        continue

    birth = birth_from_rrn(rrn)
    if birth is None:
        log(row, emp_id, '주민등록번호', rrn, '날짜 파싱 불가')
        df.at[row, '주민등록번호'] = '미입력'
        rrn_err += 1
        continue

    if not (MIN_AGE <= calc_age(birth) <= MAX_AGE):
        log(row, emp_id, '주민등록번호', rrn, f'생년월일 범위 초과 (나이={calc_age(birth)})')
        df.at[row, '주민등록번호'] = '미입력'
        rrn_err += 1
        continue

    if not rrn_checksum_ok(rrn):
        log(row, emp_id, '주민등록번호', rrn, '체크섬 오류')
        df.at[row, '주민등록번호'] = '미입력'
        rrn_err += 1
        continue

    valid_rrn[row] = birth

print('주민등록번호 정제 완료!')
print(f'  유효한 주민번호: {len(valid_rrn)}건')
print(f'  오류 (미입력 처리): {rrn_err}건')

In [ ]:
print('성별 정제 시작...')
gender_fix = 0

for row in df.index:
    rrn_digits = str(df.at[row, '주민등록번호']).replace('-', '')

    if pd.notna(df.at[row, '성별']):
        current_gender = str(df.at[row, '성별']).strip()
    else:
        current_gender = ''

    if rrn_digits != '미입력' and len(rrn_digits) >= 7 and rrn_digits.isdigit():
        if int(rrn_digits[6]) in (1, 3):
            correct_gender = '남'
        else:
            correct_gender = '여'
        if current_gender != correct_gender:
            log(row, get_emp_id(row), '성별', current_gender, f'주민번호와 불일치 → {correct_gender} 교정')
            df.at[row, '성별'] = correct_gender
            gender_fix += 1
    elif current_gender not in ('남', '여'):
        log(row, get_emp_id(row), '성별', current_gender, '결측/이상치 + 주민번호 없음')
        df.at[row, '성별'] = '미입력'
        gender_fix += 1

print('성별 정제 완료!')
print(f'  교정/미입력 처리: {gender_fix}건')
print(f'\n성별 분포:')
print(df['성별'].value_counts())

In [ ]:
print('생년월일 정제 시작...')
birth_fix = 0

for row in df.index:
    rrn_birth = valid_rrn.get(row)

    if pd.notna(df.at[row, '생년월일']):
        birth_str = str(df.at[row, '생년월일']).strip()
    else:
        birth_str = ''

    birth = parse_date(birth_str)

    if rrn_birth:
        if birth is None or not (MIN_AGE <= calc_age(birth) <= MAX_AGE):
            if birth is not None:
                log(row, get_emp_id(row), '생년월일', birth_str, '범위 초과 → 주민번호로 교정')
            df.at[row, '생년월일'] = rrn_birth.strftime('%Y-%m-%d')
            birth_fix += 1
        else:
            df.at[row, '생년월일'] = birth.strftime('%Y-%m-%d')
    else:
        if birth is None:
            log(row, get_emp_id(row), '생년월일', birth_str, '결측/파싱불가 + 주민번호 없음')
            df.at[row, '생년월일'] = '미입력'
            birth_fix += 1
        elif not (MIN_AGE <= calc_age(birth) <= MAX_AGE):
            log(row, get_emp_id(row), '생년월일', birth_str, '범위 초과')
            df.at[row, '생년월일'] = '미입력'
            birth_fix += 1
        else:
            df.at[row, '생년월일'] = birth.strftime('%Y-%m-%d')

print('생년월일 정제 완료!')
print(f'  교정/미입력 처리: {birth_fix}건')

In [ ]:
print('나이 정제 시작...')
age_fix = 0

for row in df.index:
    birth_str = str(df.at[row, '생년월일'])

    if birth_str != '미입력':
        birth = parse_date(birth_str)
    else:
        birth = None

    if birth:
        df.at[row, '나이'] = calc_age(birth)
    elif valid_rrn.get(row):
        df.at[row, '나이'] = calc_age(valid_rrn[row])
    else:
        log(row, get_emp_id(row), '나이', df.at[row, '나이'], '생년월일·주민번호 없음')
        df.at[row, '나이'] = '미입력'
        age_fix += 1

print('나이 정제 완료!')
print(f'  미입력 처리: {age_fix}건')
valid_ages = pd.to_numeric(df['나이'], errors='coerce').dropna()
print(f'\n나이 통계:')
print(f'  평균: {valid_ages.mean():.1f}세  |  최소: {valid_ages.min():.0f}세  |  최대: {valid_ages.max():.0f}세')

In [ ]:
print('병역 정제 시작...')
mil_fix = 0

for row in df.index:
    gender = str(df.at[row, '성별']).strip()

    if pd.notna(df.at[row, '병역']):
        military = str(df.at[row, '병역']).strip()
    else:
        military = ''

    if gender == '여':
        df.at[row, '병역'] = '해당없음'
    elif not military:
        log(row, get_emp_id(row), '병역', military, '결측')
        df.at[row, '병역'] = '미입력'
        mil_fix += 1

print('병역 정제 완료!')
print(f'  미입력 처리: {mil_fix}건')
print(f'\n병역 분포:')
print(df['병역'].value_counts())

### 2-3. 근무 정보 (입사일 · 근속기간)

In [ ]:
print('입사일 정제 시작...')
valid_hire = {}
hire_err = 0

for row, raw in df['입사일'].items():
    if pd.notna(raw):
        hire_str = str(raw).strip()
    else:
        hire_str = ''

    hire_date = parse_date(hire_str)
    emp_id = get_emp_id(row)

    birth_str = str(df.at[row, '생년월일'])
    if birth_str != '미입력':
        birth = parse_date(birth_str)
    else:
        birth = None

    if birth:
        earliest_hire = date(birth.year + MIN_AGE, birth.month, birth.day)
    else:
        earliest_hire = None

    if hire_date is None:
        log(row, emp_id, '입사일', hire_str, '결측/파싱불가')
        df.at[row, '입사일'] = '미입력'
        hire_err += 1
    elif hire_date > TODAY:
        log(row, emp_id, '입사일', hire_str, '현재 날짜 초과')
        df.at[row, '입사일'] = '미입력'
        hire_err += 1
    elif earliest_hire and hire_date < earliest_hire:
        log(row, emp_id, '입사일', hire_str, '만 18세 이전')
        df.at[row, '입사일'] = '미입력'
        hire_err += 1
    else:
        valid_hire[row] = hire_date
        df.at[row, '입사일'] = hire_date.strftime('%Y-%m-%d')

print('입사일 정제 완료!')
print(f'  유효한 입사일: {len(valid_hire)}건')
print(f'  오류 (미입력 처리): {hire_err}건')

In [ ]:
print('근속기간 정제 시작...')
tenure_err = 0

for row in df.index:
    hire_date = valid_hire.get(row)
    if hire_date:
        df.at[row, '근속기간'] = tenure_years(hire_date)
    else:
        log(row, get_emp_id(row), '근속기간', df.at[row, '근속기간'], '입사일 없어 계산 불가')
        df.at[row, '근속기간'] = '미입력'
        tenure_err += 1

print('근속기간 정제 완료!')
print(f'  미입력 처리: {tenure_err}건')
valid_tenure = pd.to_numeric(df['근속기간'], errors='coerce').dropna()
print(f'\n근속기간 통계:')
print(f'  평균: {valid_tenure.mean():.1f}년  |  최소: {valid_tenure.min():.1f}년  |  최대: {valid_tenure.max():.1f}년')

### 2-4. 학력 · 경력

In [ ]:
print('학력/출신대학 정제 시작...')
edu_fix = 0

for row in df.index:
    if pd.notna(df.at[row, '학력']):
        edu = str(df.at[row, '학력']).strip()
    else:
        edu = ''

    if pd.notna(df.at[row, '출신대학']):
        univ = str(df.at[row, '출신대학']).strip()
    else:
        univ = ''

    emp_id = get_emp_id(row)

    # 대학원졸 → 미입력
    if edu == '대학원졸':
        log(row, emp_id, '학력', edu, '대학원졸 허용 안 함')
        edu = '미입력'
        df.at[row, '학력'] = '미입력'
        edu_fix += 1

    # 학력 결측 → 출신대학으로 파생
    if not edu or edu == '미입력':
        if univ and univ != '미입력':
            if '전문' in univ:
                edu = '전문대졸'
            else:
                edu = '대졸'
            df.at[row, '학력'] = edu
        else:
            log(row, emp_id, '학력', edu, '결측 + 출신대학 없음')
            df.at[row, '학력'] = '미입력'
            edu = '미입력'
            edu_fix += 1

    # 출신대학 검증
    if edu == '고졸':
        if univ:
            log(row, emp_id, '출신대학', univ, '고졸인데 출신대학 존재 → 삭제')
            df.at[row, '출신대학'] = ''
    elif edu in ('대졸', '전문대졸'):
        if not univ:
            log(row, emp_id, '출신대학', univ, '결측')
            df.at[row, '출신대학'] = '미입력'
            edu_fix += 1
        else:
            is_junior_college = '전문' in univ
            if edu == '전문대졸' and not is_junior_college:
                log(row, emp_id, '출신대학', univ, '학력=전문대졸인데 일반대학교명')
                df.at[row, '출신대학'] = '미입력'
                edu_fix += 1
            elif edu == '대졸' and is_junior_college:
                log(row, emp_id, '출신대학', univ, '학력=대졸인데 전문대학교명')
                df.at[row, '출신대학'] = '미입력'
                edu_fix += 1

print('학력/출신대학 정제 완료!')
print(f'  교정/미입력 처리: {edu_fix}건')
print(f'\n학력 분포:')
print(df['학력'].value_counts())

In [ ]:
print('학점 · 채용경로 · 계약형태 · 이전직장 정제 시작...')

# 학점
for row in df.index:
    edu = str(df.at[row, '학력']).strip()
    gpa_raw = df.at[row, '학점']

    if edu == '고졸':
        df.at[row, '학점'] = ''
        continue

    gpa = to_num(gpa_raw, '학점', row, get_emp_id(row))
    if gpa is None:
        df.at[row, '학점'] = '미입력'
    elif not (MIN_GPA <= gpa <= MAX_GPA):
        log(row, get_emp_id(row), '학점', gpa_raw, f'범위 초과 ({MIN_GPA}~{MAX_GPA})')
        df.at[row, '학점'] = '미입력'

# 채용경로 / 계약형태
for col in ['채용경로', '계약형태']:
    for row, raw in df[col].items():
        if pd.notna(raw):
            cell_value = str(raw).strip()
        else:
            cell_value = ''
        if not cell_value:
            log(row, get_emp_id(row), col, cell_value, '결측')
            df.at[row, col] = '미입력'

# 이전직장명 / 이전최종직급 / 이전담당업무
for row in df.index:
    hire_route = str(df.at[row, '채용경로']).strip()
    for col in ['이전직장명', '이전최종직급', '이전담당업무']:
        if pd.notna(df.at[row, col]):
            cell_value = str(df.at[row, col]).strip()
        else:
            cell_value = ''
        if hire_route == '경력' and not cell_value:
            log(row, get_emp_id(row), col, cell_value, '채용경로=경력인데 결측')
            df.at[row, col] = '미입력'

print('학점 · 채용경로 · 계약형태 · 이전직장 정제 완료!')
print(f'\n채용경로 분포:')
print(df['채용경로'].value_counts())

### 2-5. 조직 정보 (부서 · 팀 · 직급 · 직책 · 레벨)

In [ ]:
# 회사명 / 사업장위치
for col in ['회사명', '사업장위치']:
    for row, raw in df[col].items():
        if pd.notna(raw):
            cell_value = str(raw).strip()
        else:
            cell_value = ''
        if not cell_value:
            log(row, get_emp_id(row), col, cell_value, '결측')
            df.at[row, col] = '미입력'

# 부서
print('부서/팀/부서레벨 정제 시작...')
for row, raw in df['부서'].items():
    if pd.notna(raw):
        dept = str(raw).strip()
    else:
        dept = ''
    if not dept or dept not in DEPARTMENTS:
        log(row, get_emp_id(row), '부서', dept, '결측/이상치 → 행 제거')
        drop_rows.add(row)

# 팀
for row in df.index:
    dept = str(df.at[row, '부서']).strip()
    if pd.notna(df.at[row, '팀']):
        team = str(df.at[row, '팀']).strip()
    else:
        team = ''
    if not team:
        log(row, get_emp_id(row), '팀', team, '결측')
        df.at[row, '팀'] = '미입력'
    elif dept in DEPT_TEAM_MAP and team not in DEPT_TEAM_MAP[dept]:
        log(row, get_emp_id(row), '팀', team, f'부서({dept})-팀 매핑 불일치')
        df.at[row, '팀'] = '미입력'

# 부서레벨
for row in df.index:
    dept = str(df.at[row, '부서']).strip()
    if dept in DEPT_LEVEL_MAP:
        df.at[row, '부서레벨'] = DEPT_LEVEL_MAP[dept]
    else:
        log(row, get_emp_id(row), '부서레벨', df.at[row, '부서레벨'], '부서 없음 → 행 제거')
        drop_rows.add(row)

print('부서/팀/부서레벨 정제 완료!')
print(f'\n부서 분포:')
print(df['부서'].value_counts())

In [ ]:
print('직급/직책/직급레벨 정제 시작...')

# 직급
for row, raw in df['직급'].items():
    if pd.notna(raw):
        grade = str(raw).strip()
    else:
        grade = ''
    if not grade or grade not in GRADE_LEVEL_MAP:
        log(row, get_emp_id(row), '직급', grade, '결측/이상치 → 행 제거')
        drop_rows.add(row)

# 직책
ceo_row = None
for row in df.index:
    grade = str(df.at[row, '직급']).strip()
    if pd.notna(df.at[row, '직책']):
        position = str(df.at[row, '직책']).strip()
    else:
        position = ''

    if not position or position not in POSITIONS:
        log(row, get_emp_id(row), '직책', position, '결측/이상치')
        df.at[row, '직책'] = '미입력'
        continue

    if grade == '사장' and position != '대표이사':
        log(row, get_emp_id(row), '직책', position, '직급=사장인데 직책≠대표이사 → 교정')
        df.at[row, '직책'] = '대표이사'
        position = '대표이사'

    if position == '대표이사':
        if ceo_row is None:
            ceo_row = row
        else:
            log(row, get_emp_id(row), '직책', position, '대표이사 중복 (1명만 허용)')
            df.at[row, '직책'] = '미입력'

# 직급레벨
for row in df.index:
    grade = str(df.at[row, '직급']).strip()
    if grade in GRADE_LEVEL_MAP:
        df.at[row, '직급레벨'] = GRADE_LEVEL_MAP[grade]
    else:
        log(row, get_emp_id(row), '직급레벨', df.at[row, '직급레벨'], '직급 없음 → 행 제거')
        drop_rows.add(row)

print('직급/직책/직급레벨 정제 완료!')
print(f'\n직급 분포:')
print(df['직급'].value_counts())

### 2-6. 퇴직 · 연락처 · 급여

In [ ]:
print('퇴직 정보 정제 시작...')
retire_fix = 0

for row in df.index:
    if pd.notna(df.at[row, '퇴직구분']):
        retire_type = str(df.at[row, '퇴직구분']).strip()
    else:
        retire_type = ''

    if pd.notna(df.at[row, '퇴직일자']):
        retire_date_str = str(df.at[row, '퇴직일자']).strip()
    else:
        retire_date_str = ''

    hire_date = valid_hire.get(row)
    retire_date = parse_date(retire_date_str)

    if retire_date_str and retire_date is None:
        log(row, get_emp_id(row), '퇴직일자', retire_date_str, '날짜 파싱 불가')
        df.at[row, '퇴직일자'] = '미입력'
        retire_date_str = ''
        retire_fix += 1

    if retire_date:
        if hire_date and retire_date < hire_date:
            log(row, get_emp_id(row), '퇴직일자', retire_date_str, '입사일 이전')
            df.at[row, '퇴직일자'] = '미입력'
            retire_date = None
            retire_date_str = ''
            retire_fix += 1
        elif retire_date > TODAY:
            log(row, get_emp_id(row), '퇴직일자', retire_date_str, '현재 날짜 초과')
            df.at[row, '퇴직일자'] = '미입력'
            retire_date = None
            retire_date_str = ''
            retire_fix += 1

    if retire_date and not retire_type:
        log(row, get_emp_id(row), '퇴직구분', retire_type, '퇴직일자 있는데 퇴직구분 없음')
        df.at[row, '퇴직구분'] = '미입력'
        retire_fix += 1

    if retire_type and not retire_date_str:
        log(row, get_emp_id(row), '퇴직일자', retire_date_str, '퇴직구분 있는데 퇴직일자 없음')
        df.at[row, '퇴직일자'] = '미입력'
        retire_fix += 1

print('퇴직 정보 정제 완료!')
print(f'  오류 처리: {retire_fix}건')

In [ ]:
print('연락처 정제 시작...')
email_pat = re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')
phone_pat = re.compile(r'^0\d{1,2}-\d{3,4}-\d{4}$')
seen_email = {}
contact_fix = 0

# 이메일
for row, raw in df['이메일'].items():
    if pd.notna(raw):
        email = str(raw).strip()
    else:
        email = ''

    if not email:
        log(row, get_emp_id(row), '이메일', email, '결측')
        df.at[row, '이메일'] = '미입력'
        contact_fix += 1
    elif not email_pat.match(email):
        log(row, get_emp_id(row), '이메일', email, '형식 오류')
        df.at[row, '이메일'] = '미입력'
        contact_fix += 1
    elif email in seen_email:
        log(row, get_emp_id(row), '이메일', email, '중복')
        df.at[row, '이메일'] = '미입력'
        contact_fix += 1
    else:
        seen_email[email] = row

# 전화번호
for row, raw in df['전화번호'].items():
    if pd.notna(raw):
        phone = str(raw).strip()
    else:
        phone = ''

    if not phone:
        log(row, get_emp_id(row), '전화번호', phone, '결측')
        df.at[row, '전화번호'] = '미입력'
        contact_fix += 1
    elif not phone_pat.match(phone):
        log(row, get_emp_id(row), '전화번호', phone, '형식 오류')
        df.at[row, '전화번호'] = '미입력'
        contact_fix += 1

# 주소
for row, raw in df['주소'].items():
    if pd.notna(raw):
        address = str(raw).strip()
    else:
        address = ''
    if not address:
        log(row, get_emp_id(row), '주소', address, '결측')
        df.at[row, '주소'] = '미입력'
        contact_fix += 1

print('연락처 정제 완료!')
print(f'  오류 처리: {contact_fix}건')

In [ ]:
print('급여 정보 정제 시작...')
salary_fix = 0

# 연봉
for row, raw in df['연봉'].items():
    salary = to_num(raw, '연봉', row, get_emp_id(row))
    if salary is None:
        df.at[row, '연봉'] = '미입력'
        salary_fix += 1
    elif not (MIN_SALARY <= salary <= MAX_SALARY):
        log(row, get_emp_id(row), '연봉', raw, f'범위 초과 ({MIN_SALARY:,}~{MAX_SALARY:,})')
        df.at[row, '연봉'] = '미입력'
        salary_fix += 1

# 잔업시간
for row, raw in df['잔업시간'].items():
    overtime = to_num(raw, '잔업시간', row, get_emp_id(row))
    if overtime is None:
        df.at[row, '잔업시간'] = '미입력'
        salary_fix += 1
    elif not (0 <= overtime <= MAX_OVERTIME):
        log(row, get_emp_id(row), '잔업시간', raw, f'범위 초과 (0~{MAX_OVERTIME})')
        df.at[row, '잔업시간'] = '미입력'
        salary_fix += 1

# 미사용휴가일수
for row, raw in df['미사용휴가일수'].items():
    unused_leave = to_num(raw, '미사용휴가일수', row, get_emp_id(row))
    if unused_leave is None:
        df.at[row, '미사용휴가일수'] = '미입력'
        salary_fix += 1
    elif not (0 <= unused_leave <= MAX_UNUSED_LEAVE):
        log(row, get_emp_id(row), '미사용휴가일수', raw, f'범위 초과 (0~{MAX_UNUSED_LEAVE})')
        df.at[row, '미사용휴가일수'] = '미입력'
        salary_fix += 1

# 급여은행
for row, raw in df['급여은행'].items():
    if pd.notna(raw):
        bank = str(raw).strip()
    else:
        bank = ''
    if not bank:
        log(row, get_emp_id(row), '급여은행', bank, '결측')
        df.at[row, '급여은행'] = '미입력'
        salary_fix += 1
    elif only_special(bank):
        log(row, get_emp_id(row), '급여은행', bank, '숫자/특수문자만으로 구성')
        df.at[row, '급여은행'] = '미입력'
        salary_fix += 1

# 계좌번호
for row, raw in df['계좌번호'].items():
    if pd.notna(raw):
        account = str(raw).strip()
    else:
        account = ''
    if not account:
        log(row, get_emp_id(row), '계좌번호', account, '결측')
        df.at[row, '계좌번호'] = '미입력'
        salary_fix += 1

# 4대보험가입여부
for row, raw in df['4대보험가입여부'].items():
    if pd.notna(raw):
        insurance = str(raw).strip()
    else:
        insurance = ''
    if not insurance or insurance not in INSURANCE_VALUES:
        log(row, get_emp_id(row), '4대보험가입여부', insurance, '결측/이상치')
        df.at[row, '4대보험가입여부'] = '미입력'
        salary_fix += 1

print('급여 정보 정제 완료!')
print(f'  오류 처리: {salary_fix}건')
valid_salary = pd.to_numeric(df['연봉'], errors='coerce').dropna()
print(f'\n연봉 통계:')
print(f'  평균: {valid_salary.mean():,.0f}원  |  최소: {valid_salary.min():,.0f}원  |  최대: {valid_salary.max():,.0f}원')

### 2-7. 성과 정보 (성과점수 · 인사고과)

In [ ]:
print('성과 정보 정제 시작...')
perf_fix = 0

# 성과점수
for row, raw in df['성과점수'].items():
    score = to_num(raw, '성과점수', row, get_emp_id(row))
    if score is None:
        df.at[row, '성과점수'] = '미입력'
        perf_fix += 1
    elif not (0 <= score <= MAX_SCORE):
        log(row, get_emp_id(row), '성과점수', raw, f'범위 초과 (0~{MAX_SCORE})')
        df.at[row, '성과점수'] = '미입력'
        perf_fix += 1

# 인사고과_2020~2024
for year in PERF_YEARS:
    col = f'인사고과_{year}'
    if col not in df.columns:
        continue

    for row in df.index:
        if pd.notna(df.at[row, col]):
            grade_val = str(df.at[row, col]).strip()
        else:
            grade_val = ''

        hire_date = valid_hire.get(row)
        hire_year = hire_date.year if hire_date else None

        if pd.notna(df.at[row, '퇴직일자']):
            retire_date_str = str(df.at[row, '퇴직일자']).strip()
        else:
            retire_date_str = ''
        retire_date = parse_date(retire_date_str)
        retire_year = retire_date.year if retire_date else None

        # 입사 전 연도 → 빈값
        if hire_year and year < hire_year:
            df.at[row, col] = ''
            continue

        # 퇴직 후 연도 → 빈값
        if retire_year and year > retire_year:
            df.at[row, col] = ''
            continue

        # 입사 연도 → 있으면 고정값 검증, 없으면 허용
        if hire_year and year == hire_year:
            if grade_val and grade_val not in PERF_GRADES:
                log(row, get_emp_id(row), col, grade_val, '고정값 외')
                df.at[row, col] = '미입력'
                perf_fix += 1
            continue

        # 재직 기간 내 → 필수
        if not grade_val:
            log(row, get_emp_id(row), col, grade_val, '재직 기간 내 결측')
            df.at[row, col] = '미입력'
            perf_fix += 1
        elif grade_val not in PERF_GRADES:
            log(row, get_emp_id(row), col, grade_val, '고정값 외')
            df.at[row, col] = '미입력'
            perf_fix += 1

print('성과 정보 정제 완료!')
print(f'  오류 처리: {perf_fix}건')
print(f'\n인사고과_2024 분포:')
print(df['인사고과_2024'].value_counts())

### 2-8. 자격 · 이력 (자격증 · TOEIC · 포상이력 · 징계이력)

In [ ]:
print('자격/이력 정제 시작...')
misc_fix = 0

# TOEIC점수
for row, raw in df['TOEIC점수'].items():
    if pd.notna(raw):
        toeic_str = str(raw).strip()
    else:
        toeic_str = ''
    if not toeic_str:
        continue  # 빈값 허용
    toeic = to_num(raw, 'TOEIC점수', row, get_emp_id(row))
    if toeic is None:
        df.at[row, 'TOEIC점수'] = '미입력'
        misc_fix += 1
    elif not (0 <= toeic <= MAX_TOEIC):
        log(row, get_emp_id(row), 'TOEIC점수', raw, f'범위 초과 (0~{MAX_TOEIC})')
        df.at[row, 'TOEIC점수'] = '미입력'
        misc_fix += 1

# 자격증수당여부
for row, raw in df['자격증수당여부'].items():
    if pd.notna(raw):
        subsidy = str(raw).strip()
    else:
        subsidy = ''
    if not subsidy or subsidy not in SUBSIDY_VALUES:
        log(row, get_emp_id(row), '자격증수당여부', subsidy, '결측/이상치')
        df.at[row, '자격증수당여부'] = '미입력'
        misc_fix += 1

# 자격증 / 포상이력 / 징계이력
for col in ['자격증', '포상이력', '징계이력']:
    for row, raw in df[col].items():
        all_items = parse_array(raw)
        cleaned_items = []
        removed_items = []
        for item in all_items:
            if only_special(item):
                removed_items.append(item)
            else:
                cleaned_items.append(item)
        if removed_items:
            log(row, get_emp_id(row), col, ','.join(removed_items), '숫자/특수문자만 구성 항목 제거')
            misc_fix += 1
        df.at[row, col] = ','.join(cleaned_items)

# 징계사유
for row in df.index:
    discipline_history = str(df.at[row, '징계이력']).strip()
    if pd.notna(df.at[row, '징계사유']):
        discipline_reason = str(df.at[row, '징계사유']).strip()
    else:
        discipline_reason = ''

    has_discipline = bool(discipline_history)

    if has_discipline and not discipline_reason:
        log(row, get_emp_id(row), '징계사유', discipline_reason, '징계이력 있는데 징계사유 없음')
        df.at[row, '징계사유'] = '미입력'
        misc_fix += 1
    elif not has_discipline and discipline_reason:
        log(row, get_emp_id(row), '징계사유', discipline_reason, '징계이력 없는데 징계사유 존재')
        df.at[row, '징계사유'] = ''
        misc_fix += 1

print('자격/이력 정제 완료!')
print(f'  오류 처리: {misc_fix}건')

# 3. 행 제거 적용

In [ ]:
print('행 제거 시작...')
print(f'  전체 행 수: {len(df):,}')
print(f'  제거 대상: {len(drop_rows)}행')
print('-' * 40)

df_clean = df.drop(index=list(drop_rows))
df_clean = df_clean.reset_index(drop=True)

print('행 제거 완료!')
print(f'  정제 후 행 수: {len(df_clean):,}')
print(f'  제거된 행 수: {len(df) - len(df_clean):,}')

# 4. 결과 저장 및 요약

In [ ]:
OUTPUT_PATH = OUTPUT_DIR / '통합인사정보_정제.csv'
ERROR_PATH  = OUTPUT_DIR / 'error_log.csv'

df_clean.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'정제 CSV 저장 완료!')
print(f'  저장 경로: {OUTPUT_PATH}')

if _errors:
    df_errors = pd.DataFrame(_errors)
    df_errors.to_csv(ERROR_PATH, index=False, encoding='utf-8-sig')
    print(f'\n에러 로그 저장 완료!')
    print(f'  저장 경로: {ERROR_PATH}')
    print(f'  총 에러 건수: {len(_errors):,}건')
else:
    print('\n에러 없음')

In [ ]:
print('=' * 60)
print('통합인사정보 데이터 전처리 완료')
print('=' * 60)
print(f'  원본 행 수      : {len(df):,}')
print(f'  정제 후 행 수   : {len(df_clean):,}')
print(f'  제거된 행 수    : {len(df) - len(df_clean):,}')
print(f'  총 에러 로그    : {len(_errors):,}건')
print('-' * 60)

if _errors:
    df_errors = pd.DataFrame(_errors)
    print(f'\n에러 많은 컬럼 Top 10:')
    print('-' * 40)
    print(df_errors['컬럼'].value_counts().head(10))
    print('-' * 40)
    print(f'\n에러 사유 분포:')
    print(df_errors['사유'].value_counts().head(10))

In [ ]:
print('정제 결과 미리보기:')
print('-' * 50)
df_clean.head()